# 02 — Test Configuration & First GRPO Training (Unsloth + vLLM)

Versione accelerata del notebook 01 con **Unsloth** (2-5x training più veloce, ~50-70% meno VRAM)
e **vLLM** (generazione batch fino a 10-24x più veloce durante GRPO).

> **⚠️ Requisiti:** `uv pip install -e ".[fast]"` (oppure `bash setup_linux.sh` su Colab)
>
> **Fix `OutStream` AttributeError:** Unsloth **deve** essere importato come primissima cosa,
> prima di torch e di qualsiasi altra libreria che rediriga stdout/stderr.

1. **Unsloth import first** — importa Unsloth prima di tutto
2. **Imports & GPU** — verifica che torch veda la GPU
3. **Dataset** — genera (o carica) il dataset sintetico
4. **Model** — carica il modello con Unsloth + LoRA + 4-bit
5. **Rewards** — test rapido delle reward functions
6. **Inference test** — genera un paio di completions e valuta
7. **GRPO Training** — avvia un mini-train (pochi step) con vLLM

In [14]:
!rm -rf tris

## Setup su Colab

In [15]:
!git clone https://github.com/GiuseppeBellamacina/tris.git
%cd tris
!bash setup_linux.sh

## 1. Unsloth import (deve essere il primo import)

Unsloth patcha i flussi di I/O al momento dell'import. Se viene importato **dopo** che
ipykernel ha già sostituito `sys.stdout`/`sys.stderr` con un `OutStream`, si ottiene
l'`AttributeError: 'OutStream' object has no attribute 'watch_fd_thread'`.
La soluzione è importarlo qui, come **prima cella eseguita**.

In [ ]:
# DEVE essere il primo import — prima di torch, transformers, ecc.
from unsloth import FastLanguageModel  # noqa: F401

print("Unsloth importato correttamente")


## 2. Imports & GPU Check

In [3]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: Nessuna GPU trovata! Il training sarà molto lento.")

In [5]:
import accelerate
import peft
import transformers
import trl

import datasets as hf_datasets

print(f"transformers: {transformers.__version__}")
print(f"trl:          {trl.__version__}")
print(f"peft:         {peft.__version__}")
print(f"datasets:     {hf_datasets.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print("\nTutti gli import OK")

## 3. Genera / Carica Dataset Sintetico

In [6]:
import os
from pathlib import Path

from src.datasets.dataloader import load_synthetic_dataset
from src.datasets.synthetic_dataset import generate_dataset

ROOT = Path.cwd()  # la root del progetto è la cartella corrente

os.chdir(ROOT)  # assicurati di essere nella root

DATASET_PATH = ROOT / "data" / "synthetic"

if not DATASET_PATH.exists():
    print("Dataset non trovato, lo genero...")
    ds = generate_dataset(num_samples=500, seed=42)  # 500 per test veloce
    ds.save_to_disk(str(DATASET_PATH))
    print(f"Dataset salvato in {DATASET_PATH}")
else:
    print(f"Dataset trovato in {DATASET_PATH}")

ds = load_synthetic_dataset(str(DATASET_PATH))
print(f"\nSplits: {list(ds.keys())}")
for split_name, split_ds in ds.items():
    print(f"  {split_name}: {len(split_ds)} samples")
    print(f"    columns: {split_ds.column_names}")

In [7]:
# Guarda qualche sample
train_ds = ds["train"]
for i in range(3):
    sample = train_ds[i]
    print(f"--- Sample {i} ---")
    print(f"  task_type:  {sample['task_type']}")
    print(f"  difficulty: {sample['difficulty']}")
    print(f"  prompt:     {sample['prompt'][:120]}...")
    print()

## 4. Carica Modello con Unsloth + LoRA + 4-bit

In [8]:
from src.datasets.dataloader import load_config

config = load_config(str(ROOT / "experiments" / "configs" / "grpo.yaml"))
print("Config caricata:")
for section, values in config.items():
    print(f"  [{section}]")
    if isinstance(values, dict):
        for k, v in values.items():
            print(f"    {k}: {v}")

In [9]:
from src.models.model_loader import load_model_and_tokenizer

# Forza Unsloth nel config
config["model"]["use_unsloth"] = True

print(f"Caricamento modello con Unsloth: {config['model']['name']}")
print("(questo può richiedere qualche minuto al primo download...)\n")

model, tokenizer = load_model_and_tokenizer(config)

# Info modello
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n[Unsloth] Parametri totali:    {total:,}")
print(f"[Unsloth] Parametri trainable: {trainable:,} ({100 * trainable / total:.2f}%)")
print(f"Tokenizer vocab:     {len(tokenizer)}")
print(f"Pad token:           {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

## 5. Test Reward Functions

In [10]:
from src.training.rewards import (
    combined_reward,
    json_reward,
    python_reward,
    reasoning_reward,
)

# Test JSON reward
valid_json = '```json\n{"name": "test", "age": 25, "active": true}\n```'
invalid_json = '```json\n{"name": "test", age: 25}\n```'
no_json = "Non ho generato nessun JSON."

print("=== JSON Reward ===")
print(f"  Valid JSON:   {json_reward(valid_json)}")  # atteso: 1.0
print(f"  Invalid JSON: {json_reward(invalid_json)}")  # atteso: 0.0
print(f"  No JSON:      {json_reward(no_json)}")  # atteso: 0.0

# Test Python reward
valid_py = "```python\ndef add(a, b):\n    return a + b\n```"
invalid_py = "```python\ndef add(a, b)\n    return a + b\n```"  # manca :
no_py = "Ecco la mia risposta senza codice."

print("\n=== Python Reward ===")
print(f"  Valid Python:   {python_reward(valid_py)}")  # atteso: 1.0
print(f"  Invalid Python: {python_reward(invalid_py)}")  # atteso: 0.0
print(f"  No Python:      {python_reward(no_py)}")  # atteso: 0.0

# Test reasoning bonus
with_think = "<think>Devo creare un JSON con tre campi come richiesto.</think>\n" + valid_json
without_think = valid_json

print("\n=== Reasoning Reward ===")
print(f"  With <think>:    {reasoning_reward(with_think)}")  # atteso: 0.2
print(f"  Without <think>: {reasoning_reward(without_think)}")  # atteso: 0.0

# Test combined
print("\n=== Combined Reward ===")
print(f"  JSON valid:           {combined_reward(valid_json, 'json')}")  # 1.0
r = combined_reward(with_think, "json", reasoning_bonus=0.2)
print(f"  JSON valid + reason:  {r}")  # 1.2

print("\nReward functions OK")

## 6. Test Inference (pre-training)

In [11]:
from src.datasets.dataloader import format_prompt_for_model

# Prendi 4 sample misti
test_indices = [0, 1, 10, 20]
test_samples = [train_ds[i] for i in test_indices if i < len(train_ds)]

print(f"Genero completions per {len(test_samples)} prompt...\n")

for sample in test_samples:
    prompt = format_prompt_for_model(sample, tokenizer)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    completion = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

    # Valuta
    task_type = sample["task_type"]
    difficulty = sample["difficulty"]
    reward = combined_reward(completion, task_type=task_type)

    print(f"--- [{task_type}/{difficulty}] ---")
    print(f"Prompt:     {sample['prompt'][:100]}...")
    print(f"Completion: {completion[:200]}")
    print(f"Reward:     {reward}")
    print()

## 7. Mini GRPO Training con Unsloth + vLLM (pochi step di test)

Facciamo un training ridotto per verificare che la pipeline giri senza errori.

- **max_steps=20** — solo per test
- **num_generations=4** — minimo per avere varianza nelle reward
- **learning_rate=1e-6** — basso per evitare model collapse
- **beta=0.1** — KL penalty più alto per restare vicini al modello base
- **50 samples** — subset piccolo
- **use_vllm=True** — generazione batch accelerata durante GRPO

In [12]:
from datasets import Dataset
from src.training.grpo_train import build_reward_fn

# Usa solo 50 samples per il test
MINI_N = 50
mini_ds = train_ds.select(range(min(MINI_N, len(train_ds))))  # type: ignore[arg-type]

# Formatta i prompt
formatted = []
for i in range(len(mini_ds)):
    s = mini_ds[i]
    p = format_prompt_for_model(s, tokenizer)
    formatted.append({"prompt": p})

prompt_dataset = Dataset.from_list(formatted)
print(f"Mini dataset: {len(prompt_dataset)} prompt pronti")

# Reward function
reward_fn = build_reward_fn(config["reward"])
print("Reward function creata")

In [ ]:
import datetime

from trl.trainer.grpo_config import GRPOConfig

# Config leggera per test
mini_output_dir = str(ROOT / "experiments" / "checkpoints" / "grpo_test_fast")
mini_log_dir = str(ROOT / "experiments" / "logs" / "grpo_test_fast")
Path(mini_output_dir).mkdir(parents=True, exist_ok=True)
Path(mini_log_dir).mkdir(parents=True, exist_ok=True)

_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"grpo-test-unsloth-vllm-{_ts}"

# bf16 richiede GPU Ampere+ (compute capability >= 8.0) con CUDA >= 11.0;
# su GPU più vecchie (es. T4/V100) si ricade su fp16.
_supports_bf16 = (
    torch.cuda.is_available()
    and torch.cuda.get_device_capability()[0] >= 8
)
use_bf16 = _supports_bf16
use_fp16 = not _supports_bf16 and torch.cuda.is_available()
print(f"bf16={use_bf16}, fp16={use_fp16}  "
      f"(GPU compute capability: {torch.cuda.get_device_capability() if torch.cuda.is_available() else 'N/A'})")

grpo_config = GRPOConfig(  # type: ignore[call-arg]
    output_dir=mini_output_dir,
    run_name=run_name,
    project="grpo-test-01",
    # Training ridotto
    max_steps=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-6,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    bf16=use_bf16,
    fp16=use_fp16,
    # GRPO
    num_generations=4,  # minimo per avere varianza con reward binarie
    max_completion_length=256,
    beta=0.1,  # KL penalty più alto per evitare divergenza
    temperature=0.7,
    # vLLM — accelera la generazione dei completions durante il training
    use_vllm=True,
    # logging
    logging_dir=mini_log_dir,
    logging_steps=1,
    report_to="wandb",
)

print("GRPOConfig creata")
print(f"  run_name:           {grpo_config.run_name}")
print(f"  max_steps:          {grpo_config.max_steps}")
print(f"  num_generations:    {grpo_config.num_generations}")
print(f"  batch_size:         {grpo_config.per_device_train_batch_size}")
print(f"  grad_accum:         {grpo_config.gradient_accumulation_steps}")
print(f"  learning_rate:      {grpo_config.learning_rate}")
print(f"  beta (KL penalty):  {grpo_config.beta}")
print(f"  use_vllm:           {grpo_config.use_vllm}")
print(f"  logging_steps:      {grpo_config.logging_steps}")
print(f"  report_to:          {grpo_config.report_to}")


In [ ]:
from trl.trainer.grpo_trainer import GRPOTrainer

# Ricarica il modello fresco (il precedente potrebbe aver accumulato stato)
# Se hai poca VRAM, decommenta le 3 righe sotto per liberare memoria prima
# del model
torch.cuda.empty_cache()
model, tokenizer = load_model_and_tokenizer(config)

trainer = GRPOTrainer(  # type: ignore[arg-type]
    model=model,
    args=grpo_config,
    train_dataset=prompt_dataset,
    reward_funcs=reward_fn,
    processing_class=tokenizer,
)

print("GRPOTrainer inizializzato")
print("\nAvvio mini-training (20 step)...\n")

trainer.train()
print("\nMini-training completato")

## 8. Verifica post-training

In [ ]:
# Rigenera sugli stessi prompt di prima per vedere se qualcosa è cambiato
print("Completions dopo il mini-training:\n")

for sample in test_samples:
    prompt = format_prompt_for_model(sample, tokenizer)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    completion = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

    task_type = sample["task_type"]
    difficulty = sample["difficulty"]
    reward = combined_reward(completion, task_type=task_type)

    print(f"--- [{task_type}/{difficulty}] ---")
    print(f"Prompt:     {sample['prompt'][:100]}...")
    print(f"Completion: {completion[:200]}")
    print(f"Reward:     {reward}")
    print()

In [ ]:
# Pulizia VRAM
del trainer
del model
torch.cuda.empty_cache()
print("VRAM liberata")

**Accelerazione attiva in questo notebook:**
- **Unsloth**: 2-5x training più veloce, ~50-70% meno VRAM
- **vLLM**: inferenza batch fino a 10-24x più veloce (generazione GRPO)

**Prossimi passi:**
- Training completo: `uv run python -m src.training.grpo_train --config experiments/configs/grpo.yaml`
- Baseline eval: `uv run python -m src.evaluation.baseline_eval --config experiments/configs/baseline.yaml`
- SFT comparison: `uv run python -m src.training.sft_train --config experiments/configs/sft.yaml`